In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split as surprise_split
import joblib

print("="*70)
print("PHASE 3: HYBRID RECOMMENDATION SYSTEM")
print("="*70)

# ============================================================
# STEP 1: LOAD ALL FEATURES FROM PHASE 2
# ============================================================
print("\n[1/6] Loading features from Phase 2...")

df_train = pd.read_csv('train_data.csv')
item_features = pd.read_csv('item_features.csv')
user_profiles = pd.read_csv('user_profiles.csv')

# Load feature matrices
tfidf_matrix = np.load('tfidf_matrix.npy')
lda_matrix = np.load('lda_matrix.npy')
doc_embeddings = np.load('doc_embeddings.npy')

# Load trained models
with open('tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
with open('lda_model.pkl', 'rb') as f:
    lda_model = pickle.load(f)
with open('w2v_model.pkl', 'rb') as f:
    w2v_model = pickle.load(f)

print(f"✓ Loaded {len(df_train)} training reviews")
print(f"✓ Loaded {len(item_features)} items with features")
print(f"✓ Loaded {len(user_profiles)} user profiles")

# ============================================================
# STEP 2: BUILD COLLABORATIVE FILTERING (SVD)
# ============================================================
print("\n[2/6] Training Collaborative Filtering (SVD)...")

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_train[['user_id', 'item_id', 'rating']], reader)
trainset, testset = surprise_split(data, test_size=0.2, random_state=42)

svd_model = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(trainset)

# Get CF predictions on test set
cf_predictions = svd_model.test(testset)
print(f"✓ SVD model trained: {len(trainset.all_ratings())} ratings")
print(f"✓ Test set size: {len(testset)} ratings")

# ============================================================
# STEP 3: BUILD CONTENT-BASED FEATURES
# ============================================================
print("\n[3/6] Building content-based feature vectors...")

# Feature 1: Sentiment-based similarity
sentiment_features = item_features[['sentiment_polarity', 'sentiment_subjectivity']].fillna(0).values
sentiment_scaler = StandardScaler()
sentiment_norm = sentiment_scaler.fit_transform(sentiment_features)

# Feature 2: Topic distribution
topic_features = item_features[[f'topic_{i}' for i in range(15)]].fillna(0).values

# Feature 3: Aspect sentiments
aspect_features = item_features[[f'aspect_{asp}' for asp in 
                                 ['plot', 'characters', 'writing_style', 'setting', 'emotion', 'pacing']]].fillna(0).values

# Feature 4: Embeddings (already normalized)
embedding_features = item_features[[f'embedding_{i}' for i in range(100)]].fillna(0).values

# Combine all features (weighted)
combined_content = np.hstack([
    sentiment_norm * 0.1,      # Weight: 10%
    topic_features * 0.2,      # Weight: 20%
    aspect_features * 0.3,     # Weight: 30%
    embedding_features * 0.4   # Weight: 40%
])

# Normalize combined features
content_scaler = StandardScaler()
content_features_norm = content_scaler.fit_transform(combined_content)

print(f"✓ Content-based features: {content_features_norm.shape}")
print(f"  - Sentiment: 10%")
print(f"  - Topics: 20%")
print(f"  - Aspects: 30%")
print(f"  - Embeddings: 40%")

# ============================================================
# STEP 4: BUILD ITEM-ITEM SIMILARITY MATRIX
# ============================================================
print("\n[4/6] Computing item-item similarity matrix...")

# Compute cosine similarity between items
item_similarity = cosine_similarity(content_features_norm)
print(f"✓ Item similarity matrix: {item_similarity.shape}")

# ============================================================
# STEP 5: HYBRID RECOMMENDATION ENGINE
# ============================================================
print("\n[5/6] Building hybrid recommendation engine...")

class HybridRecommender:
    def __init__(self, svd_model, item_similarity, item_features, 
                 user_profiles, df_train, alpha=0.5):
        """
        Hybrid recommender combining CF and content-based
        
        alpha: 0 = pure content-based, 1 = pure CF
               0.5 = equal weight
        """
        self.svd_model = svd_model
        self.item_similarity = item_similarity
        self.item_features = item_features
        self.user_profiles = user_profiles
        self.df_train = df_train
        self.alpha = alpha
        
        # Create item_id to index mapping
        self.item_id_to_idx = {iid: idx for idx, iid in enumerate(item_features['item_id'].values)}
        self.user_id_to_idx = {uid: idx for idx, uid in enumerate(user_profiles['user_id'].values)}
        
        # Items user has already rated
        self.user_items = df_train.groupby('user_id')['item_id'].apply(set).to_dict()
    
    def get_cf_score(self, user_id, item_id):
        """Get collaborative filtering score"""
        try:
            return self.svd_model.predict(user_id, item_id).est
        except:
            return 3.0  # Default rating
    
    def get_content_score(self, user_id, item_id):
        """Get content-based score using user-item similarity"""
        if user_id not in self.user_id_to_idx or item_id not in self.item_id_to_idx:
            return 3.0
        
        item_idx = self.item_id_to_idx[item_id]
        user_rated_items = self.user_items.get(user_id, set())
        
        if not user_rated_items:
            return 3.0
        
        # Get user's rated items indices
        user_item_indices = [self.item_id_to_idx[iid] for iid in user_rated_items 
                            if iid in self.item_id_to_idx]
        
        if not user_item_indices:
            return 3.0
        
        # Average similarity to user's rated items
        similarities = self.item_similarity[item_idx][user_item_indices]
        
        # Weight by user ratings
        user_ratings = {}
        for _, row in self.df_train[self.df_train['user_id'] == user_id].iterrows():
            if row['item_id'] in self.item_id_to_idx:
                user_ratings[self.item_id_to_idx[row['item_id']]] = row['rating']
        
        weighted_sim = sum(similarities[i] * user_ratings.get(idx, 3.0) 
                          for i, idx in enumerate(user_item_indices))
        avg_sim = weighted_sim / len(user_item_indices)
        
        return np.clip(avg_sim, 1, 5)
    
    def recommend(self, user_id, n_recommendations=10, exclude_rated=True):
        """Generate hybrid recommendations"""
        
        if user_id not in self.user_id_to_idx:
            # Cold-start user: use item popularity
            return self._cold_start_recommend(n_recommendations)
        
        recommendations = []
        
        for item_id in self.item_features['item_id']:
            # Skip already-rated items
            if exclude_rated and item_id in self.user_items.get(user_id, set()):
                continue
            
            # Get scores from both methods
            cf_score = self.get_cf_score(user_id, item_id)
            content_score = self.get_content_score(user_id, item_id)
            
            # Hybrid score: weighted average
            hybrid_score = self.alpha * cf_score + (1 - self.alpha) * content_score
            
            recommendations.append({
                'item_id': item_id,
                'hybrid_score': hybrid_score,
                'cf_score': cf_score,
                'content_score': content_score
            })
        
        # Sort and return top N
        recommendations = sorted(recommendations, key=lambda x: x['hybrid_score'], reverse=True)
        return recommendations[:n_recommendations]
    
    def _cold_start_recommend(self, n_recommendations=10):
        """Recommend popular items for cold-start users"""
        popular = self.item_features.nlargest(n_recommendations, 'review_count')[['item_id', 'avg_rating']]
        return [{'item_id': iid, 'hybrid_score': rating, 'cf_score': 3.0, 'content_score': rating}
                for iid, rating in zip(popular['item_id'], popular['avg_rating'])]

# Initialize recommender with balanced weights
recommender = HybridRecommender(
    svd_model=svd_model,
    item_similarity=item_similarity,
    item_features=item_features,
    user_profiles=user_profiles,
    df_train=df_train,
    alpha=0.5  # 50% CF, 50% content-based
)

print("✓ Hybrid recommender initialized (α=0.5)")

# ============================================================
# STEP 6: EVALUATE ON TEST SET
# ============================================================
print("\n[6/6] Evaluating recommendations...")

from sklearn.metrics import mean_squared_error, mean_absolute_error

# Get CF predictions
cf_scores = [pred.est for pred in cf_predictions]
actual_ratings = [pred.r_ui for pred in cf_predictions]

cf_rmse = np.sqrt(mean_squared_error(actual_ratings, cf_scores))
cf_mae = mean_absolute_error(actual_ratings, cf_scores)

print(f"\nCollaborative Filtering Performance:")
print(f"  - RMSE: {cf_rmse:.4f}")
print(f"  - MAE: {cf_mae:.4f}")

# Test hybrid on sample
sample_users = df_train['user_id'].unique()[:100]
print(f"\nSample Hybrid Recommendations (first 5 users):")

for user_id in sample_users[:5]:
    recs = recommender.recommend(user_id, n_recommendations=5)
    print(f"\n  User {user_id}:")
    for i, rec in enumerate(recs, 1):
        print(f"    {i}. Item {rec['item_id']}: {rec['hybrid_score']:.2f} "
              f"(CF: {rec['cf_score']:.2f}, Content: {rec['content_score']:.2f})")

# ============================================================
# STEP 7: SAVE MODELS AND ENGINE
# ============================================================
print("\n" + "="*70)
print("SAVING MODELS AND RECOMMENDER ENGINE")
print("="*70)

joblib.dump(svd_model, 'svd_model.pkl')
joblib.dump(recommender, 'hybrid_recommender.pkl')
joblib.dump(item_similarity, 'item_similarity.pkl')

np.save('content_features_norm.npy', content_features_norm)

print("✓ svd_model.pkl")
print("✓ hybrid_recommender.pkl")
print("✓ item_similarity.pkl")
print("✓ content_features_norm.npy")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("PHASE 3 COMPLETE!")
print("="*70)
print("\nHybrid Recommendation System Built:")
print("  ✓ Collaborative Filtering (SVD) - captures user-item patterns")
print("  ✓ Content-Based Features - sentiment, topics, aspects, embeddings")
print("  ✓ Adaptive Weighting - combines both signals")
print("\nModel Performance:")
print(f"  - CF RMSE: {cf_rmse:.4f}")
print(f"  - CF MAE: {cf_mae:.4f}")
print("\nReady for Phase 4: Evaluation & Optimization!")